# setting up retraining pipelines

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a baseline classifier with scikit-learn
- Evaluate using accuracy + confusion matrix
- Show how to encode categorical features

## 🔗 Prerequisites

- ✅ Python basics
- ✅ Jupyter Notebook basics

---

## Official Structure Reference

This notebook covers practical activities from **Course 11, Unit 5**:
- setting up retraining pipelines
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md`

---


## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# small synthetic dataset
rng = np.random.default_rng(123)
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
color = rng.choice(['red','green','blue'], size=n)

y = ((x1 + 0.8*x2 + (color == 'red')*0.6 + rng.normal(scale=0.5, size=n)) > 0.2).astype(int)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'color': color, 'y': y})
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
 ('cat', OneHotEncoder(handle_unknown='ignore'), ['color']),
], remainder='passthrough')

clf = Pipeline([
 ('pre', pre),
 ('model', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print('confusion matrix:', confusion_matrix(y_test, y_pred))
print('\nreport:', classification_report(y_test, y_pred))


## 🌍 Real-World Application: Automated Retraining Pipeline

Models decay as real-world data evolves. Below is a complete automated retraining pipeline similar to what Spotify uses for its recommendation system.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time

np.random.seed(42)
print("=== Automated Retraining Pipeline Demo ===")
print("Simulating Spotify-style model retraining\n")

data = load_breast_cancer()
X_base, y_base = data.data[:400], data.target[:400]
X_new_pool, y_new_pool = data.data[400:], data.target[400:]

# Initial model
X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_base, test_size=0.2, random_state=42)
current_model = RandomForestClassifier(n_estimators=50, random_state=42)
current_model.fit(X_tr, y_tr)
baseline_acc = accuracy_score(y_te, current_model.predict(X_te))
print(f"Initial model accuracy: {baseline_acc:.3f}")

RETRAIN_THRESHOLD = baseline_acc - 0.03  # retrain if accuracy drops by 3%

log = [{'week': 0, 'accuracy': baseline_acc, 'retrained': False, 'data_size': len(X_base)}]

# Simulate 8 weeks of production
X_cumulative, y_cumulative = X_tr.copy(), y_tr.copy()
for week in range(1, 9):
    # New data arrives weekly (with slight distribution shift)
    n_new = 20
    idx = np.random.choice(len(X_new_pool), n_new, replace=False)
    X_week = X_new_pool[idx] * (1 + 0.05*week*np.random.randn(*X_new_pool[idx].shape))  # slight shift
    y_week = y_new_pool[idx]
    
    # Evaluate current model on new data
    week_acc = accuracy_score(y_week, current_model.predict(X_week))
    retrained = False
    
    if week_acc < RETRAIN_THRESHOLD:
        # Trigger retraining
        X_cumulative = np.vstack([X_cumulative, X_week])
        y_cumulative = np.append(y_cumulative, y_week)
        current_model = RandomForestClassifier(n_estimators=50, random_state=42+week)
        current_model.fit(X_cumulative, y_cumulative)
        week_acc = accuracy_score(y_week, current_model.predict(X_week))
        retrained = True
        print(f"  Week {week}: RETRAIN TRIGGERED — new acc={week_acc:.3f}, data_size={len(X_cumulative)}")
    else:
        print(f"  Week {week}: No retrain needed — acc={week_acc:.3f}")
    
    log.append({'week': week, 'accuracy': week_acc, 'retrained': retrained, 'data_size': len(X_cumulative)})

# Plot retraining log
fig, ax = plt.subplots(figsize=(10, 4))
weeks = [l['week'] for l in log]
accs  = [l['accuracy'] for l in log]
retrain_weeks = [l['week'] for l in log if l['retrained']]
ax.plot(weeks, accs, 'o-', color='steelblue', label='Model accuracy')
ax.axhline(RETRAIN_THRESHOLD, color='red', linestyle='--', label='Retrain threshold')
for w in retrain_weeks:
    ax.axvline(w, color='orange', alpha=0.5, linewidth=3, label='Retrain event' if w==retrain_weeks[0] else '')
ax.set_xlabel("Week"); ax.set_ylabel("Accuracy"); ax.set_title("Automated Retraining Pipeline")
ax.legend(); plt.tight_layout(); plt.savefig('/tmp/retraining_pipeline.png', dpi=72)
print("\nRetraining pipeline complete.")
print("Real-world: Spotify retrains recommendation models hourly with this exact trigger approach.")

## 📝 Summary

You set up a **CI/CD pipeline** for ML models — automatically testing, validating, and deploying on every code push. MLOps = DevOps + ML-specific concerns (data drift, model decay, experiment tracking). Companies like Netflix and Google re-deploy models hundreds of times per day using these pipelines.